# Structure Prediction

This notebook demonstrates the `predict_structures` dispatch function, which provides a unified entry point for running any structure prediction tool in proto-tools. Rather than importing a specific tool like ESMFold or Protenix directly, you pass the tool name as a string and `predict_structures` handles input construction, dispatch, and output normalization. This pattern is particularly useful when the choice of backend is determined at runtime or configured by a user. The example below predicts a protein-ligand complex — RuvB (a bacterial helicase) bound to ADP — using Boltz2.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("boltz2")
display_overview("boltz2")
display_docs_section("boltz2", "Background")

# Boltz-2

Boltz-2 is an openly licensed biomolecular structure prediction model from the [MIT Jameel Clinic](https://jclinic.mit.edu/) and [Recursion](https://www.recursion.com/), built in the AlphaFold3 family: a diffusion model that predicts the joint 3D structure of complexes mixing proteins, DNA, RNA, and small-molecule ligands, together with the binding affinity of a small molecule against a protein target. This toolkit runs Boltz-2 structure and affinity prediction on a local GPU, with optional ColabFold multiple-sequence alignments.

Boltz-2 ([Passaro et al., 2025](https://doi.org/10.1101/2025.06.14.659707)) predicts the joint 3D structure of a biomolecular assembly from the sequences and chemical components it contains. It builds on Boltz-1, one of the most widely used open-source alternatives to AlphaFold3, extending that co-folding model with a binding-affinity module, improved controllability, and additional training data. Like AlphaFold3, a single model folds complexes that mix proteins, DNA, RNA, and small-molecule ligands and predicts how those components are arranged relative to one another. Each protein chain can be paired with a multiple-sequence alignment (MSA) of evolutionarily related sequences, whose covariation patterns supply the evolutionary signal the model uses to place residues.

Architecturally, Boltz-2 reproduces AlphaFold3: it carries a single representation of the input tokens and a pairwise representation over token pairs, refines them through an AlphaFold3-style trunk, and generates all-atom coordinates with a diffusion module that starts from noise and iteratively denoises into a structure. Several structures can be sampled per complex and ranked by a confidence score, reported as a complex predicted local distance difference test (pLDDT) for local reliability, a predicted aligned error (PAE) for the relative placement of any two tokens, and predicted template-modeling (pTM) and interface predicted template-modeling (ipTM) scores that summarize overall and interface accuracy. Beyond structure, Boltz-2 adds a binding-affinity module that approaches the accuracy of physics-based free-energy perturbation while running more than 1000 times faster.

The reference implementation is open-sourced at [jwohlwend/boltz](https://github.com/jwohlwend/boltz) under the MIT license, covering the code, weights, and training pipeline for both academic and commercial use, with the released weights distributed as `boltz-community/boltz-2`. It was developed by the Boltz team at the [MIT Jameel Clinic](https://jclinic.mit.edu/) together with [Recursion](https://www.recursion.com/).

## Available tools

In [2]:
# Show available functions for the structure prediction tools demonstrated in this notebook
display_available_tools("boltz2")

- **`run_boltz2_affinity()`** — Predicted binding affinity (log10 IC50 μM) and binder probability for a small molecule against a protein target, via Boltz-2.
- **`run_boltz2()`** — Multi-modal structure prediction using Boltz2

### `predict_structures`

The `predict_structures` dispatch function accepts one or more `Complex` objects and a `toolkit` string, then routes the request to the appropriate tool implementation. This makes it straightforward to swap backends — ESMFold for speed, Boltz2 or Protenix for multi-modal accuracy — without changing any input construction code. Each complex is a list of chains with sequences and entity types. Protein chains use amino acid sequences, while ligands use SMILES notation.

In [3]:
from proto_tools.tools.structure_prediction.dispatch import predict_structures
from proto_tools.tools.structure_prediction.shared_data_models import Complex

In [4]:
# Display input API reference for Boltz2 (the tool used in this example)
display_api_reference("boltz2", "input", "run_boltz2")

# RuvB helicase sequence (bacterial, from Thermus thermophilus)
RuvB = "VERTLRPQYFKEYIGQDKVKDQLKIFIEAAKLRDEALDHTLLFGPPGLGKTTMAFVIANEMGVNLKQTSGPAIEKAGDLVAILNDLEPGDILFIDEIHRMPMAVEEVLYSAMEDYYIDIMIGAGETSRSVHLDLPPFTLVGATTRAGMLSNPLRARFGINGHMEYYELPDLTEIVERTSEIFEMTITPEAALELARRSRGTPRIANRLLKRVRDYAQIMGDGVIDDKIADQALTMLDVDHEGLDYVDQKILRTMIEMYGGGPVGLGTLSVNIAEERETVEDMYEPYLIQKGFIMRTRTGRVATAKAYEHMGYDYTRDN"

# ADP (adenosine diphosphate) SMILES
ADP = "Nc1ncnc2c1ncn2[C@@H]1O[C@H](CO[P@](=O)([O-])OP(=O)([O-])[O-])[C@@H](O)[C@H]1O"

# Create protein-ligand complex
input_complex = Complex(
    chains=[
        {"sequence": RuvB, "entity_type": "protein"},
        {"sequence": ADP, "entity_type": "ligand"},
    ]
)

**Input** — `Boltz2Input`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `complexes` | `list[proto_tools.entities.complex.Complex]` | required | List of complexes to predict structure for containing chains and entity types. |
| `msas` | `list[proto_tools.tools.structure_prediction.shared_data_models.ComplexMSAs] \| None` | `None` | Per-complex MSAs; a bare dict[int, MSA] is coerced to an unpaired ComplexMSAs. |

In [5]:
# Display config API reference for Boltz2
display_api_reference("boltz2", "config", "run_boltz2")

# Tool config can be passed as a dict; predict_structures will instantiate the correct config class
tool_config = {"verbose": True}

**Config** — `Boltz2Config`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `verbose` | `int` | `0` | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| `device` | `str` | `'cuda'` | Device to run the model on (e.g., 'cuda', 'cpu') |
| `timeout` | `int \| None` | `1200` | Maximum execution time in seconds. |
| `seed` | `int \| None` | `None` | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| `include_pae_matrix` | `bool` | `False` | Attach the full per-residue PAE matrix. |
| `use_msa` | `bool` | `True` | Whether to generate and use MSAs for protein chains using MMseqs2 homology search |
| `msa_search_config` | `proto_tools.tools.sequence_alignment.mmseqs2.homology_search.Mmseqs2HomologySearchConfig \| None` | `None` | Nested MMseqs2 homology-search config for MSA generation; None uses default settings. |
| `recycling_steps` | `int` | `3` | Iterative refinement passes through the model. Higher = more accurate but slower. |
| `sampling_steps` | `int` | `200` | Denoising steps in the diffusion process. Higher = more refined but slower. |
| `diffusion_samples` | `int` | `1` | Structure samples per complex; best by confidence is kept. Higher = more thorough but slower. |
| `step_scale` | `float` | `1.5` | Diffusion step size (typical range 1.0-2.0). Lower values produce more sample diversity. |
| `max_msa_seqs` | `int` | `8192` | Maximum number of MSA sequences fed into the model. Lower this to reduce GPU memory on deep MSAs. |
| `subsample_msa` | `bool` | `False` | Randomly subsample the MSA on each run for sample diversity (loses determinism). |
| `num_workers` | `int` | `4` | Number of dataloader workers for prediction. |

In [6]:
# Run the prediction using the dispatch interface
output = predict_structures(
    complexes=input_complex,
    toolkit="boltz2",
    tool_config=tool_config,
)

Generating MSAs: 1 unpaired sequence(s), 0 paired group(s) for heterocomplexes...


Running remote_msa_search.py (one-shot) with device=cpu


Running run_mmseqs2_homology_search [00:00]

Folding structures (Boltz-2):   0%|          | 0/1 [00:00<?, ?complex/s]

Using local GPU for Boltz2 structure prediction...


Assigned MSA to chain A (3578 sequences)


In [7]:
# Display output API reference for Boltz2
display_api_reference("boltz2", "output", "run_boltz2")

structure = output.structures[0]

# Print basic structure metrics
print(f"Chain IDs:          {structure.get_chain_ids()}")
print(f"Number of residues: {structure.num_residues}")
print(f"Confidence score:   {structure.metrics['confidence_score']:.3f}")
print(f"Complex pLDDT:      {structure.metrics['complex_plddt']:.3f}")
print(f"pTM score:          {structure.metrics['ptm']:.3f}")
print(f"ipTM score:         {structure.metrics['iptm']:.3f}")

# Visualize the complex colored by chain to distinguish protein from ligand
structure.visualize(color_by="chain", style="ribbon")

**Output** — `Boltz2Output`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `structures` | `list[proto_tools.entities.structures.structure.Structure]` | required | List of predicted structures, one per input complex. |

Chain IDs:          ['A']
Number of residues: 318
Confidence score:   0.889
Complex pLDDT:      0.866
pTM score:          0.906
ipTM score:         0.981


## Export Results

Predicted structures returned by `predict_structures` can be exported to PDB or mmCIF format for downstream analysis in molecular visualization tools such as PyMOL, ChimeraX, or VMD, and as input to further computational workflows. The B-factor column contains per-residue pLDDT confidence scores.

In [8]:
from pathlib import Path

# Create output directory
output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

# Export the predicted structure to PDB
output.export(name="ruvb_adp_complex", export_path=output_dir, file_format="pdb")

proto-tools/proto_tools/entities/structures/structure.py:569: UserWarning: CIF→PDB conversion: 1 residue name(s) exceed PDB's 3-character limit and will be truncated (e.g., ['LIG1']).
  return convert_cif_str_to_pdb_str(self.structure)
